# 15 Scaling Study Up To 40 Qubits

Estimate memory, runtime, Hamiltonian diagonal size, grid resolution, and QFT gate counts without allocating impossible statevectors.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
qubits = config["quantum"]["scaling_qubits"]
resources = resource_estimate_table(qubits)
save_table(resources, "15_resource_scaling_up_to_40_qubits.csv")
mem40_128 = int(resources.loc[resources["grid_qubits"] == 40, "complex128_bytes"].iloc[0])
mem40_64 = int(resources.loc[resources["grid_qubits"] == 40, "complex64_bytes"].iloc[0])
assert mem40_128 > 17_000_000_000_000 and mem40_64 > 8_000_000_000_000
print("VALIDATION PASSED: 40-qubit statevector memory is estimated only, not allocated")
plt.figure()
plt.semilogy(resources["grid_qubits"], resources["complex128_bytes"] / 1e12, marker="o", label="complex128 TB")
plt.semilogy(resources["grid_qubits"], resources["complex64_bytes"] / 1e12, marker="o", label="complex64 TB")
plt.axhline(float(config["quantum"]["safe_statevector_memory_gb"]) / 1000, color="black", linestyle="--", label="default safe limit")
plt.title("Statevector memory scaling up to 40 qubits")
plt.xlabel("Grid qubits")
plt.ylabel("Memory (TB, decimal)")
plt.legend()
save_current_figure("15_resource_scaling_memory.png")
plt.figure()
plt.semilogy(resources["grid_qubits"], resources["runtime_proxy_N_log2N"], marker="o")
plt.title("Runtime proxy N log2 N")
plt.xlabel("Grid qubits")
plt.ylabel("Proxy units")
save_current_figure("15_resource_scaling_runtime_proxy.png")
plt.figure()
plt.plot(resources["grid_qubits"], resources["qft_two_qubit_gate_estimate"], marker="o", label="two-qubit")
plt.plot(resources["grid_qubits"], resources["small_circuit_depth_estimate"], marker="o", label="depth proxy")
plt.title("QFT gate and depth estimates")
plt.xlabel("Grid qubits")
plt.ylabel("Gate/depth estimate")
plt.legend()
save_current_figure("15_qft_gate_depth_estimates.png")
resources
